In [ ]:
# examples/run_mpc.py
import torch
from citylearn.citylearn import CityLearnEnv
from citylearn_evmodel.agents.mpc import MPC
from my_models.lstm import LSTM  # import your LSTM class

# 1) Build env
env = CityLearnEnv(schema="path/to/your/schema.json")

# 2) Load trained LSTM (weights saved in Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
lstm_model = LSTM(
    n_features=13, n_output=1, seq_len=24, num_hidden=64, num_layers=2, drop_prob=0.2, weight_decay=1e-4
).to(device)
state = torch.load("my_lstm_model.pth", map_location=device)
lstm_model.load_state_dict(state)
lstm_model.eval()

# 3) Create agent
agent = MPC(
    env,
    horizon=6,
    comfort_temp=22.0,
    comfort_range=(20.0, 24.0),
    alpha=1.0, beta=2.0,
    lstm_model=lstm_model,
    num_features=13,
    seq_len=24,
    control_building_index=0,
    setpoint_action_index=0,  # adjust to your action definition
)

# 4) Rollout
obs, info = env.reset()
terminated = False
while not terminated:
    actions = agent.predict(obs, deterministic=True)
    obs, rewards, terminated, truncated, info = env.step(actions)

print("Training completed successfully!")
